In [ ]:
"""
Find incomplete tasks with two-phase analysis - NOTEBOOK VERSION

Phase 1: Check pd/ and lgd/ result files to determine completion status
Phase 2: For missing tasks only, analyze log files to determine failure causes

Location: Run from scripts/Experiment1/ directory
Results: Located in results/experiment1/
"""

import sys
import re
import pickle
from pathlib import Path
from collections import defaultdict
from datetime import datetime

# Auto-detect project root from scripts/Experiment1/
def find_project_root():
    """Find project root by looking for config/ directory."""
    current = Path.cwd()
    
    # If we're in scripts/Experiment1/, go up 2 levels
    if current.name == 'Experiment1' and current.parent.name == 'scripts':
        return current.parent.parent
    
    # If we're in scripts/, go up 1 level
    if current.name == 'scripts':
        return current.parent
    
    # Try current directory
    if (current / 'config').exists():
        return current
    
    # Search upward
    for parent in current.parents:
        if (parent / 'config').exists():
            return parent
    
    raise FileNotFoundError("Could not find project root (no config/ directory found)")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

# Add project to path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import project modules
from src.utils.config_reader import load_config
from src.utils.storage_handler import StorageHandler
from src.methods.method_config import NO_HPO_METHODS


def get_expected_tasks(config):
    """Build list of all expected tasks based on Experiment1 config."""
    tasks = []
    
    # PD tasks
    pd_datasets = [ds for ds, enabled in config['datasets']['pd'].items() if enabled]
    pd_methods = [m for m, enabled in config['methods']['pd'].items() if enabled]
    
    for dataset in pd_datasets:
        for method in pd_methods:
            tasks.append((dataset, method, 'pd', 'NO_HPO'))
            tasks.append((dataset, method, 'pd', 'HPO'))
    
    # LGD tasks
    lgd_datasets = [ds for ds, enabled in config['datasets']['lgd'].items() if enabled]
    lgd_methods = [m for m, enabled in config['methods']['lgd'].items() if enabled]
    
    for dataset in lgd_datasets:
        for method in lgd_methods:
            tasks.append((dataset, method, 'lgd', 'NO_HPO'))
            tasks.append((dataset, method, 'lgd', 'HPO'))
    
    return set(tasks)


def get_completed_tasks(experiment_dir):
    """Get list of completed tasks from result files."""
    completed = []
    
    for task_type in ['pd', 'lgd']:
        task_dir = experiment_dir / task_type
        
        if not task_dir.exists():
            print(f"Warning: Directory not found: {task_dir}")
            continue
        
        for result_file in task_dir.glob('*.pkl'):
            dataset = result_file.stem
            
            try:
                with open(result_file, 'rb') as f:
                    results = pickle.load(f)
                
                if 'NO_HPO' in results:
                    for method in results['NO_HPO'].keys():
                        completed.append((dataset, method, task_type, 'NO_HPO'))
                
                if 'HPO' in results:
                    for method in results['HPO'].keys():
                        completed.append((dataset, method, task_type, 'HPO'))
            
            except Exception as e:
                print(f"Warning: Could not read {result_file}: {e}")
    
    return set(completed)


def parse_errors_log_for_tasks(experiment_dir, missing_tasks):
    """Parse errors.log ONLY for tasks that are missing."""
    errors_log = experiment_dir / "logs" / "errors.log"
    failures = {}
    
    if not errors_log.exists():
        return failures
    
    missing_keys = {(d, m, h) for d, m, t, h in missing_tasks}
    
    try:
        with open(errors_log, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        
        blocks = content.split('=' * 70)
        
        for block in blocks:
            if 'FAILED:' not in block:
                continue
            
            match = re.search(r'FAILED:\s+(.+?)/([\w_]+)/(NO_HPO|HPO)', block)
            if not match:
                continue
            
            dataset, method, hpo_mode = match.group(1), match.group(2), match.group(3)
            key = (dataset, method, hpo_mode)
            
            if key not in missing_keys:
                continue
            
            error_match = re.search(r'Error:\s*(.+?)(?=\nNode:|\Z)', block, re.DOTALL)
            time_match = re.search(r'Time:\s*(.+)', block)
            node_match = re.search(r'Node:\s*(.+)', block)
            
            error_text = error_match.group(1).strip() if error_match else 'Unknown'
            time_text = time_match.group(1).strip() if time_match else ''
            node_text = node_match.group(1).strip() if node_match else ''
            
            error_type = 'other'
            if 'CUDA out of memory' in error_text:
                error_type = 'cuda_oom'
            elif 'AssertionError' in error_text:
                error_type = 'assertion'
            elif not error_text or error_text == 'Unknown':
                error_type = 'unknown'
            
            failures[key] = {
                'error': error_text[:200],
                'type': error_type,
                'time': time_text,
                'node': node_text
            }
    
    except Exception as e:
        print(f"Warning: Could not parse errors.log: {e}")
    
    return failures


def find_timeout_tasks_for_missing(experiment_dir, missing_tasks):
    """Find SLURM timeouts ONLY for tasks that are missing."""
    slurm_logs = experiment_dir / "logs" / "slurm"
    timeouts = []
    
    if not slurm_logs.exists():
        return timeouts
    
    missing_keys = {(d, m, h) for d, m, t, h in missing_tasks}
    
    for err_file in slurm_logs.glob("*.err"):
        try:
            with open(err_file, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
            
            if 'TIME LIMIT' not in content and 'CANCELLED' not in content:
                continue
            
            out_file = err_file.with_suffix('.out')
            if not out_file.exists():
                continue
            
            with open(out_file, 'r', encoding='utf-8', errors='ignore') as f:
                out_content = f.read()
            
            match = re.search(
                r'Running task \d+:\s+(.+?)/([\w_]+)/(pd|lgd)/(NO_HPO|HPO)',
                out_content
            )
            
            if not match:
                dataset_match = re.search(r'Dataset:\s+(.+?)$', out_content, re.MULTILINE)
                method_match = re.search(r'Method:\s+([\w_]+)$', out_content, re.MULTILINE)
                task_match = re.search(r'Task type:\s+(pd|lgd)$', out_content, re.MULTILINE)
                hpo_match = re.search(r'HPO mode:\s+(NO_HPO|HPO)$', out_content, re.MULTILINE)
                
                if all([dataset_match, method_match, task_match, hpo_match]):
                    dataset = dataset_match.group(1).strip()
                    method = method_match.group(1).strip()
                    task_type = task_match.group(1).strip()
                    hpo_mode = hpo_match.group(1).strip()
                else:
                    continue
            else:
                dataset = match.group(1)
                method = match.group(2)
                task_type = match.group(3)
                hpo_mode = match.group(4)
            
            key = (dataset, method, hpo_mode)
            if key not in missing_keys:
                continue
            
            time_match = re.search(
                r'\[(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})\.\d+\]',
                content
            )
            timestamp = time_match.group(1) if time_match else 'Unknown'
            
            node_match = re.search(r'ON ([\w\d]+)', content)
            node = node_match.group(1) if node_match else 'Unknown'
            
            timeouts.append({
                'dataset': dataset,
                'method': method,
                'task_type': task_type,
                'hpo_mode': hpo_mode,
                'time': timestamp,
                'node': node,
                'err_file': err_file.name
            })
        
        except Exception as e:
            continue
    
    return timeouts


def print_section_header(title):
    """Print a formatted section header."""
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def print_subsection_header(title):
    """Print a formatted subsection header."""
    print(f"\n{title}")
    print("-" * 70)


def group_by_method(items, key_func):
    """Group items by method/hpo_mode."""
    grouped = defaultdict(list)
    for item in items:
        key = key_func(item)
        grouped[key].append(item)
    return grouped


# ============================================================
# MAIN ANALYSIS
# ============================================================

print("Loading Experiment1 configuration...")
config = load_config("Experiment1")

storage = StorageHandler("experiment1")
experiment_dir = storage.get_experiment_path()

if not experiment_dir.exists():
    print(f"\n❌ ERROR: Experiment directory not found: {experiment_dir}")
    print("Have you run any tasks yet?")
else:
    print_section_header("EXPERIMENT 1 - COMPLETION STATUS")
    print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Results: {experiment_dir}")
    
    # ============================================================
    # PHASE 1: CHECK RESULT FILES ONLY
    # ============================================================
    print_section_header("PHASE 1: CHECKING RESULT FILES")
    print("\n📊 Analyzing pd/ and lgd/ directories...")
    
    expected = get_expected_tasks(config)
    completed = get_completed_tasks(experiment_dir)
    missing = expected - completed
    
    total_expected = len(expected)
    total_completed = len(completed)
    total_missing = len(missing)
    completion_rate = (total_completed / total_expected * 100) if total_expected > 0 else 0
    
    print_section_header("SUMMARY STATISTICS")
    
    print(f"\nTotal expected tasks:  {total_expected:,}")
    print(f"Completed tasks:       {total_completed:,} ({completion_rate:.1f}%)")
    print(f"Missing tasks:         {total_missing:,} ({100-completion_rate:.1f}%)")
    
    # Break down by task type
    pd_expected = {t for t in expected if t[2] == 'pd'}
    pd_completed = {t for t in completed if t[2] == 'pd'}
    lgd_expected = {t for t in expected if t[2] == 'lgd'}
    lgd_completed = {t for t in completed if t[2] == 'lgd'}
    
    print(f"\nBy task type:")
    print(f"  PD:  {len(pd_completed):,}/{len(pd_expected):,} "
          f"({len(pd_completed)/len(pd_expected)*100:.1f}%)")
    print(f"  LGD: {len(lgd_completed):,}/{len(lgd_expected):,} "
          f"({len(lgd_completed)/len(lgd_expected)*100:.1f}%)")
    
    # Break down by HPO mode
    no_hpo_expected = {t for t in expected if t[3] == 'NO_HPO'}
    no_hpo_completed = {t for t in completed if t[3] == 'NO_HPO'}
    hpo_expected = {t for t in expected if t[3] == 'HPO'}
    hpo_completed = {t for t in completed if t[3] == 'HPO'}
    
    print(f"\nBy HPO mode:")
    print(f"  NO_HPO: {len(no_hpo_completed):,}/{len(no_hpo_expected):,} "
          f"({len(no_hpo_completed)/len(no_hpo_expected)*100:.1f}%)")
    print(f"  HPO:    {len(hpo_completed):,}/{len(hpo_expected):,} "
          f"({len(hpo_completed)/len(hpo_expected)*100:.1f}%)")
    
    if not missing:
        print_section_header("✅ ALL TASKS COMPLETED!")
        print("\n🎉 Experiment 1 has finished successfully!")
        print(f"   Total tasks completed: {total_completed:,}/{total_expected:,}")
        print("\n" + "=" * 70)
    else:
        # Show missing tasks grouped by method
        print_section_header(f"MISSING TASKS ({total_missing:,})")
        
        missing_list = sorted(missing, key=lambda x: (x[1], x[3], x[2], x[0]))
        grouped = group_by_method(missing_list, lambda x: f"{x[1]}/{x[3]}")
        
        for method_mode in sorted(grouped.keys()):
            tasks = grouped[method_mode]
            print_subsection_header(f"{method_mode} ({len(tasks)} tasks)")
            
            pd_tasks = [t for t in tasks if t[2] == 'pd']
            lgd_tasks = [t for t in tasks if t[2] == 'lgd']
            
            if pd_tasks:
                datasets = sorted([t[0] for t in pd_tasks])
                print(f"  PD ({len(pd_tasks)}): {', '.join(datasets)}")
            
            if lgd_tasks:
                datasets = sorted([t[0] for t in lgd_tasks])
                print(f"  LGD ({len(lgd_tasks)}): {', '.join(datasets)}")
        
        # ============================================================
        # PHASE 2: ANALYZE LOGS FOR MISSING TASKS ONLY
        # ============================================================
        print_section_header("PHASE 2: ANALYZING FAILURE CAUSES")
        print("\n🔍 Checking log files for missing tasks only...")
        
        python_errors = parse_errors_log_for_tasks(experiment_dir, missing)
        timeouts = find_timeout_tasks_for_missing(experiment_dir, missing)
        
        timeout_keys = {(t['dataset'], t['method'], t['hpo_mode']) for t in timeouts}
        cuda_oom_keys = {k for k, v in python_errors.items() if v['type'] == 'cuda_oom'}
        assertion_keys = {k for k, v in python_errors.items() if v['type'] == 'assertion'}
        other_error_keys = {k for k, v in python_errors.items() 
                           if v['type'] not in ['cuda_oom', 'assertion']}
        
        missing_with_task = list(missing)
        
        timeout_missing = [m for m in missing_with_task if (m[0], m[1], m[3]) in timeout_keys]
        cuda_oom_missing = [m for m in missing_with_task if (m[0], m[1], m[3]) in cuda_oom_keys]
        assertion_missing = [m for m in missing_with_task if (m[0], m[1], m[3]) in assertion_keys]
        other_error_missing = [m for m in missing_with_task if (m[0], m[1], m[3]) in other_error_keys]
        unknown_missing = [m for m in missing_with_task 
                          if (m[0], m[1], m[3]) not in timeout_keys 
                          and (m[0], m[1], m[3]) not in cuda_oom_keys
                          and (m[0], m[1], m[3]) not in assertion_keys
                          and (m[0], m[1], m[3]) not in other_error_keys]
        
        print_section_header("FAILURE BREAKDOWN")
        
        print(f"\nCategorized failures:")
        print(f"  SLURM timeouts:      {len(timeout_missing):,}")
        print(f"  CUDA out of memory:  {len(cuda_oom_missing):,}")
        print(f"  Assertion errors:    {len(assertion_missing):,}")
        print(f"  Other Python errors: {len(other_error_missing):,}")
        print(f"  Unknown/Not started: {len(unknown_missing):,}")
        
        # Display each category
        if timeout_missing:
            print_section_header(f"⏱️  SLURM TIMEOUTS ({len(timeout_missing)} tasks)")
            grouped = group_by_method(timeout_missing, lambda x: f"{x[1]}/{x[3]}")
            for method_mode in sorted(grouped.keys()):
                tasks = grouped[method_mode]
                datasets = [f"{t[0]}/{t[2]}" for t in sorted(tasks)]
                print(f"\n{method_mode} ({len(tasks)}): {', '.join(datasets)}")
            print("\n💡 Solution: Use retry scripts with increased walltime/resources")
        
        if cuda_oom_missing:
            print_section_header(f"🔥 CUDA OUT OF MEMORY ({len(cuda_oom_missing)} tasks)")
            grouped = group_by_method(cuda_oom_missing, lambda x: f"{x[1]}/{x[3]}")
            for method_mode in sorted(grouped.keys()):
                tasks = grouped[method_mode]
                datasets = [f"{t[0]}/{t[2]}" for t in sorted(tasks)]
                print(f"\n{method_mode} ({len(tasks)}): {', '.join(datasets)}")
            print("\n💡 Solution: Move to A100 (40GB) or H100 (80GB) GPUs")
        
        if assertion_missing:
            print_section_header(f"⚠️  ASSERTION ERRORS ({len(assertion_missing)} tasks)")
            grouped = group_by_method(assertion_missing, lambda x: f"{x[1]}/{x[3]}")
            for method_mode in sorted(grouped.keys()):
                tasks = grouped[method_mode]
                datasets = [f"{t[0]}/{t[2]}" for t in sorted(tasks)]
                print(f"\n{method_mode} ({len(tasks)}): {', '.join(datasets)}")
            print("\n💡 Solution: Fix method_runner.py to force tune=False for NO_HPO methods")
        
        if other_error_missing:
            print_section_header(f"❌ OTHER PYTHON ERRORS ({len(other_error_missing)} tasks)")
            grouped = group_by_method(other_error_missing, lambda x: f"{x[1]}/{x[3]}")
            for method_mode in sorted(grouped.keys()):
                tasks = grouped[method_mode]
                datasets = [f"{t[0]}/{t[2]}" for t in sorted(tasks)]
                print(f"\n{method_mode} ({len(tasks)}): {', '.join(datasets)}")
            print(f"\n💡 Solution: Check {experiment_dir / 'logs' / 'errors.log'}")
        
        if unknown_missing:
            print_section_header(f"❓ UNKNOWN/NOT STARTED ({len(unknown_missing)} tasks)")
            
            # Special handling for problematic datasets
            base_model_tasks = [m for m in unknown_missing if 'base_model' in m[0]]
            if base_model_tasks:
                problem_datasets = set([m[0] for m in base_model_tasks])
                print(f"\n⚠️  WARNING: {len(base_model_tasks)} tasks for datasets: {', '.join(problem_datasets)}")
                print("   These datasets may be missing or corrupted!")
                print("   Investigate: Check if dataset files exist and can be loaded")
            
            grouped = group_by_method(unknown_missing, lambda x: f"{x[1]}/{x[3]}")
            for method_mode in sorted(grouped.keys()):
                tasks = grouped[method_mode]
                datasets = [f"{t[0]}/{t[2]}" for t in sorted(tasks)]
                print(f"\n{method_mode} ({len(tasks)}): {', '.join(datasets)}")
            print("\n💡 These tasks have no error logs - may not have started or currently running")
        
        print_section_header("📋 NEXT STEPS")
        print("\n1. Check for problematic datasets (if applicable)")
        print("2. Submit retry jobs: sbatch Experiment1_Retry_*.slurm")
        print("3. Monitor progress: squeue -u $USER")
        print("\n" + "=" * 70)

print("\n✅ Analysis complete!")